# EDA: логика определения дефолтности по ФП/СФП

| Сегмент | ФП | СФП |
|---------|-----|-----|
| **ККБ** | 13, 13.1, 15.1.3.1 | 15.2.1, 15.2.2, 15.2.1.1, 15.2.1.2 |
| **МСБ** | 13, 15.1.3 | 15.2, 15.2.1.1 |
| **МкБ** | 13, 15.1.2 | 15.2, 15.2У, 15.2.1У |

Дефолт = появление **любого** из перечисленных факторов. `default_date` = дата первого.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from itertools import combinations

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
plt.rcParams.update({"font.family": "sans-serif",
                      "font.sans-serif": ["Arial", "DejaVu Sans"],
                      "axes.unicode_minus": False})

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")
CUTOFF    = DATE_TO
MIN_OBS_MONTHS = 3

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ", "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ", "ДКБ": "ККБ",
}
ZO_MAP = {
    "ДМСБ (микро)": "ДМ (микро)", "ДМСБ (малый)": "ДМСБ (малый)",
    "ДМСБ (средний)": "ДМСБ (средний)", "ДКБ": "ДКБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]

DEFAULT_RULES = {
    "ККБ": {"FP": {"13","13.1","15.1.3.1"}, "SFP": {"15.2.1","15.2.2","15.2.1.1","15.2.1.2"}},
    "МСБ": {"FP": {"13","15.1.3"}, "SFP": {"15.2","15.2.1.1"}},
    "МкБ": {"FP": {"13","15.1.2"}, "SFP": {"15.2","15.2У","15.2.1У"}},
}
DEFAULT_ALL = {seg: r["FP"] | r["SFP"] for seg, r in DEFAULT_RULES.items()}
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

def normalize_inn(s):
    if pd.isna(s): return None
    s = str(s).strip()
    if s.endswith(".0"): s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)

## 1. Загрузка и подготовка

In [ ]:
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df.columns = df.columns.str.strip()
print(f"Загружено: {len(df):,} строк")

if "VAL" in df.columns:
    before = len(df)
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"Фильтр по источникам: {len(df):,} (было {before:,})")

df["inn"] = df["X_INN"].apply(normalize_inn)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
df["TYPE_FP"] = df["TYPE_FP"].replace({"FP": "ФП", "SFP": "СФП"})
df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
df = df[df["segment"].notna()].copy()

before_dedup = len(df)
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации: {len(df):,} (удалено {before_dedup - len(df):,})")
print(f"Уникальных ИНН: {df['inn'].nunique():,}")

## 2. Проверка наличия факторов из правил

In [ ]:
print(f"{'Сегмент':<8} {'Тип':<6} {'Фактор':<12} {'Найден?':<10} {'Записей'}")
print("=" * 55)
missing = []
for seg in SEG_ORDER:
    seg_df = df[df["segment"] == seg]
    for role, tp in [("FP", "ФП"), ("SFP", "СФП")]:
        for fp in sorted(DEFAULT_RULES[seg][role]):
            cnt = len(seg_df[(seg_df["fp_num"] == fp) & (seg_df["TYPE_FP"] == tp)])
            ok = "Да" if cnt > 0 else "НЕТ"
            print(f"{seg:<8} {role:<6} {fp:<12} {ok:<10} {cnt:,}")
            if cnt == 0: missing.append((seg, role, fp))

if missing:
    all_fp = set(df["fp_num"].dropna().unique())
    print(f"\n⚠ Не найдено: {len(missing)}")
    for seg, role, fp in missing:
        sim = sorted([f for f in all_fp if f.startswith(fp) or fp.startswith(f)])
        print(f"  {seg}/{role}/{fp} → похожие: {sim if sim else '—'}")
else:
    print("\n✅ Все факторы найдены.")

## 3. Определение дефолта

In [ ]:
def_records = []
for seg in SEG_ORDER:
    rules = DEFAULT_RULES[seg]
    s = df[df["segment"] == seg]
    mask = ((s["fp_num"].isin(rules["FP"])) & (s["TYPE_FP"] == "ФП")) | \
           ((s["fp_num"].isin(rules["SFP"])) & (s["TYPE_FP"] == "СФП"))
    def_records.append(s[mask])
df_def_events = pd.concat(def_records, ignore_index=True)

defaults = (
    df_def_events.groupby(["inn", "segment"])
    .agg(default_date=("dt", "min"), first_factor=("fp_num", "first"),
         n_def_factors=("fp_num", "nunique"))
    .reset_index()
)
defaults["default_flag"] = 1

companies = df[["inn", "segment"]].drop_duplicates()
companies = companies.merge(
    defaults[["inn", "segment", "default_flag", "default_date"]],
    on=["inn", "segment"], how="left"
)
companies["default_flag"] = companies["default_flag"].fillna(0).astype(int)

print(f"{'Сегмент':<8} {'Всего':<12} {'Дефолт':<12} {'DR'}")
print("=" * 38)
for seg in SEG_ORDER:
    c = companies[companies["segment"] == seg]
    n, nd = len(c), c["default_flag"].sum()
    print(f"{seg:<8} {n:<12,} {nd:<12,} {nd/n*100:.2f}%")
n, nd = len(companies), companies["default_flag"].sum()
print(f"{'ИТОГО':<8} {n:<12,} {nd:<12,} {nd/n*100:.2f}%")

In [ ]:
first_factors = df_def_events.sort_values("dt").groupby(["inn", "segment"]).first().reset_index()
print("Первый определяющий фактор:\n")
for seg in SEG_ORDER:
    sf = first_factors[first_factors["segment"] == seg]
    print(f"{seg}:")
    for fp, cnt in sf["fp_num"].value_counts().items():
        tp = sf[sf["fp_num"] == fp]["TYPE_FP"].mode().iloc[0]
        print(f"  {fp:<12} ({tp})  {cnt:>6,}  ({cnt/len(sf)*100:.1f}%)")
    print()

## 3.1 Минимальное окно наблюдения

In [ ]:
windows = [("Без огр.", 0), ("≥3 мес.", 3), ("≥6 мес.", 6), ("≥12 мес.", 12)]
dwd = companies[companies["default_flag"] == 1].copy()

print(f"{'Окно':<12}", end="")
for seg in SEG_ORDER: print(f"  {seg:>16}", end="")
print(f"  {'ИТОГО':>16}")
print("=" * 75)

for label, m in windows:
    min_d = DATE_FROM + pd.DateOffset(months=m)
    row = f"{label:<12}"
    for seg in SEG_ORDER:
        sd = dwd[dwd["segment"] == seg]
        a, k = len(sd), len(sd[sd["default_date"] >= min_d])
        row += f"  {k:>5,}/{a:>5,} (-{(a-k)/a*100:.0f}%)" if a > 0 else f"  {'—':>16}"
    a = len(dwd)
    k = len(dwd[dwd["default_date"] >= min_d])
    row += f"  {k:>5,}/{a:>5,} (-{(a-k)/a*100:.0f}%)"
    print(row)

MIN_OBS_DATE = DATE_FROM + pd.DateOffset(months=MIN_OBS_MONTHS)
early_default_inns = companies[
    (companies["default_flag"]==1) & (companies["default_date"] < MIN_OBS_DATE)
]["inn"].tolist()
print(f"\nПрименяем окно {MIN_OBS_MONTHS} мес.: исключено {len(early_default_inns)} ранних дефолтов")
companies = companies[~companies["inn"].isin(early_default_inns)].copy()
defaults = defaults[~defaults["inn"].isin(early_default_inns)].copy()
df = df[~df["inn"].isin(early_default_inns)].copy()
n, nd = len(companies), int(companies["default_flag"].sum())
print(f"Компаний: {n:,} | Дефолтных: {nd:,} | DR: {nd/n*100:.2f}%")

## 4. Временной фильтр (anti-data-leakage)

In [ ]:
fp_all = df[["inn", "segment", "fp_num", "TYPE_FP", "dt"]].copy()
fp_all = fp_all.merge(companies[["inn", "segment", "default_flag", "default_date"]],
                       on=["inn", "segment"], how="inner")
fp_all["ref_date"] = fp_all["default_date"].fillna(CUTOFF)

before_filter = len(fp_all)
fp_filtered = fp_all[fp_all["dt"] < fp_all["ref_date"]].copy()
print(f"До фильтра: {before_filter:,} | После: {len(fp_filtered):,} | "
      f"Отсечено: {before_filter - len(fp_filtered):,}")

def is_defining(row):
    r = DEFAULT_RULES.get(row["segment"], {})
    return (row["TYPE_FP"] == "ФП" and row["fp_num"] in r.get("FP", set())) or \
           (row["TYPE_FP"] == "СФП" and row["fp_num"] in r.get("SFP", set()))

fp_filtered["_is_def"] = fp_filtered.apply(is_defining, axis=1)
n_excl = fp_filtered["_is_def"].sum()
fp_predictors = fp_filtered[~fp_filtered["_is_def"]].copy()
print(f"Исключено определяющих: {n_excl:,} | Предикторов: {len(fp_predictors):,}")

### 4.1 Потеря записей при временном фильтре

In [ ]:
fp_d = fp_all[fp_all["default_flag"] == 1]
fp_db = fp_d[fp_d["dt"] < fp_d["ref_date"]]
fp_da = fp_d[fp_d["dt"] >= fp_d["ref_date"]]

print(f"{'Сегмент':<8} {'Всего':<12} {'До деф.':<14} {'После деф.':<14} {'% отсеч.'}")
print("=" * 55)
for seg in SEG_ORDER:
    a = len(fp_d[fp_d["segment"]==seg])
    b = len(fp_db[fp_db["segment"]==seg])
    c = len(fp_da[fp_da["segment"]==seg])
    print(f"{seg:<8} {a:<12,} {b:<14,} {c:<14,} {c/a*100:.1f}%" if a else f"{seg}: —")
a, b, c = len(fp_d), len(fp_db), len(fp_da)
print(f"{'ИТОГО':<8} {a:<12,} {b:<14,} {c:<14,} {c/a*100:.1f}%")

print("\nСреднее ФП на компанию:")
for seg in SEG_ORDER:
    nd = companies[(companies["segment"]==seg)&(companies["default_flag"]==1)].shape[0]
    if nd == 0: continue
    print(f"  {seg}: до={len(fp_db[fp_db['segment']==seg])/nd:.1f}  "
          f"после={len(fp_da[fp_da['segment']==seg])/nd:.1f}")

## 5. Pivot (wide-формат)

In [ ]:
fp_predictors["value"] = 1
fp_wide = (fp_predictors.pivot_table(index="inn", columns="fp_num", values="value",
                                      aggfunc="sum", fill_value=0).reset_index())
fp_wide.columns.name = None

meta_cols = ["inn", "segment", "default_flag", "default_date"]
dataset = fp_wide.merge(companies[meta_cols], on="inn", how="left")
dataset["default_flag"] = dataset["default_flag"].fillna(0).astype("int8")
fp_cols = [c for c in dataset.columns if c not in meta_cols]

dataset_bin = dataset.copy()
dataset_bin[fp_cols] = dataset_bin[fp_cols].clip(upper=1).astype("int8")

dataset["fp_count"] = dataset[fp_cols].sum(axis=1)
dataset["fp_unique"] = (dataset[fp_cols] > 0).sum(axis=1)
dataset_bin["fp_count"] = dataset["fp_count"]
dataset_bin["fp_unique"] = dataset["fp_unique"]

companies_no_fp = companies[~companies["inn"].isin(dataset["inn"])].copy()
n_def_companies = int(companies["default_flag"].sum())

print(f"Датасет: {dataset.shape[0]:,} компаний × {len(fp_cols)} ФП")
print(f"Дефолтных: {int(dataset['default_flag'].sum()):,} | DR: {dataset['default_flag'].mean():.2%}")
for seg in SEG_ORDER:
    s = dataset[dataset["segment"]==seg]
    print(f"  {seg}: {len(s):,} комп., {s['default_flag'].sum():,} деф. ({s['default_flag'].mean():.2%})")

def_no_fp = companies_no_fp[companies_no_fp["default_flag"]==1]
print(f"\n'Внезапные' дефолты (нет ФП до дефолта): {len(def_no_fp)} из {n_def_companies} "
      f"({len(def_no_fp)/n_def_companies*100:.1f}%)")
for seg in SEG_ORDER:
    nl = len(def_no_fp[def_no_fp['segment']==seg])
    na = int(companies[(companies['segment']==seg)&(companies['default_flag']==1)].shape[0])
    if nl > 0: print(f"  {seg}: {nl} из {na}")

## 6. Гистограммы ФП: дефолтные vs недефолтные

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for seg, ax in zip(SEG_ORDER + ["ВСЕ"], axes.flat):
    s = dataset if seg == "ВСЕ" else dataset[dataset["segment"] == seg]
    d, nd = s[s["default_flag"]==1]["fp_unique"], s[s["default_flag"]==0]["fp_unique"]
    mx = int(s["fp_unique"].quantile(0.95)) + 1
    bins = range(0, mx + 2)
    ax.hist(nd.clip(upper=mx), bins=bins, alpha=0.7, density=True,
            label=f"Недеф. ({len(nd):,})", color="steelblue", edgecolor="white")
    ax.hist(d.clip(upper=mx), bins=bins, alpha=0.7, density=True,
            label=f"Деф. ({len(d):,})", color="tomato", edgecolor="white")
    ax.set_xlabel("Уникальных ФП"); ax.set_ylabel("Плотность")
    ax.set_title(seg, fontweight="bold"); ax.legend(fontsize=8)
plt.suptitle("Распределение числа ФП-предикторов", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Динамика дефолтов

In [ ]:
dc = companies[companies["default_flag"]==1].copy()
dc["def_month"] = dc["default_date"].dt.to_period("M")
dc["def_quarter"] = dc["default_date"].dt.to_period("Q")

monthly = dc.groupby(["def_month","segment"]).size().unstack(fill_value=0)
monthly = monthly[[s for s in SEG_ORDER if s in monthly.columns]]
monthly["Итого"] = monthly.sum(axis=1)
monthly.index = monthly.index.astype(str)
display(monthly)

colors = {"МкБ": "#4472C4", "МСБ": "#ED7D31", "ККБ": "#70AD47"}
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
for seg in SEG_ORDER:
    if seg in monthly.columns:
        axes[0].plot(monthly.index, monthly[seg], marker="o", markersize=3,
                     label=seg, color=colors[seg], linewidth=1.5)
axes[0].set_title("Дефолты по месяцам", fontweight="bold")
axes[0].set_ylabel("Кол-во ИНН"); axes[0].legend()
axes[0].tick_params(axis="x", rotation=45, labelsize=7)

quarterly = dc.groupby(["def_quarter","segment"]).size().unstack(fill_value=0)
quarterly = quarterly[[s for s in SEG_ORDER if s in quarterly.columns]]
quarterly.index = quarterly.index.astype(str)
sc = [s for s in SEG_ORDER if s in quarterly.columns]
quarterly[sc].plot(kind="bar", stacked=True, ax=axes[1],
                   color=[colors[s] for s in sc], edgecolor="white")
axes[1].set_title("Дефолты по кварталам", fontweight="bold")
axes[1].set_ylabel("Кол-во ИНН"); axes[1].legend()
plt.tight_layout(); plt.show()

## 8. Корреляции ФП (heatmap)

In [ ]:
TOP_N_HEATMAP = 30
for seg in SEG_ORDER:
    sub = dataset_bin[dataset_bin["segment"]==seg]
    if len(sub) < 10: continue
    freq = sub[fp_cols].sum().sort_values(ascending=False)
    active = freq[freq >= 2].head(TOP_N_HEATMAP).index.tolist()
    if len(active) < 2: continue

    corr = sub[active].corr()
    mask_t = np.triu(np.ones(corr.shape, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr, mask=mask_t, annot=False, cmap="RdBu_r", center=0,
                vmin=-0.5, vmax=0.5, linewidths=0.3, ax=ax)
    ax.set_title(f"Корреляции — {seg} ({len(sub):,} комп.)", fontsize=13, fontweight="bold")
    plt.xticks(fontsize=7, rotation=90); plt.yticks(fontsize=7)
    plt.tight_layout(); plt.show()

    pairs = corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool)).stack().reset_index()
    pairs.columns = ["ФП_1","ФП_2","r"]
    strong = pairs[pairs["r"].abs() > 0.3].sort_values("r", key=abs, ascending=False)
    if len(strong) > 0:
        print(f"  {seg}: |r|>0.3 — {len(strong)} пар")
        display(strong.head(10).style.hide(axis="index").format({"r": "{:.3f}"}))

## 9. Кросс-таблицы и Lift

In [ ]:
def cross_table(ds, cols):
    br = ds["default_flag"].mean()
    tot = len(ds)
    rows = []
    for c in cols:
        nw = (ds[c]==1).sum()
        if nw == 0: continue
        nd = ds.loc[ds[c]==1, "default_flag"].sum()
        r = nd/nw
        rows.append({"ФП": c, "Комп.": nw, "%": round(nw/tot*100,2),
                     "Деф.": nd, "DR%": round(r*100,2),
                     "Lift": round(r/br,2) if br>0 else 0})
    return pd.DataFrame(rows).sort_values("Lift", ascending=False), br

ct_all, br_all = cross_table(dataset_bin, fp_cols)
print(f"Общая: base DR={br_all:.2%}")
display(ct_all[ct_all["Деф."]>=3].head(30).style.hide(axis="index")
        .bar(subset=["Lift"], color="salmon"))

cross_tables = {}
for seg in SEG_ORDER:
    sd = dataset_bin[dataset_bin["segment"]==seg]
    sc = [c for c in fp_cols if (sd[c]>0).any()]
    ct, br = cross_table(sd, sc)
    cross_tables[seg] = ct
    print(f"\n{seg}: base DR={br:.2%}, компаний={len(sd):,}")
    flt = ct[ct["Деф."]>=3]
    if len(flt) > 0:
        display(flt.head(20).style.hide(axis="index").bar(subset=["Lift"], color="salmon"))

## 10. Топ комбинаций ФП по Lift

In [ ]:
for seg in SEG_ORDER:
    sd = dataset_bin[dataset_bin["segment"]==seg]
    br = sd["default_flag"].mean()
    if br == 0: continue
    y = sd["default_flag"].values
    sc = [c for c in fp_cols if (sd[c]==1).sum() >= 20]
    X = sd[sc].values; names = np.array(sc)

    pair_res = []
    for i, j in combinations(range(len(sc)), 2):
        m = (X[:,i]==1) & (X[:,j]==1)
        n = m.sum()
        if n < 10: continue
        nd = y[m].sum()
        if nd < 3: continue
        pair_res.append({"Комб.": f"{names[i]} + {names[j]}",
                         "N": n, "Деф.": nd, "DR%": round(nd/n*100,2),
                         "Lift": round(nd/n/br,2)})

    ct_seg = cross_tables.get(seg, pd.DataFrame())
    top_i = ct_seg[ct_seg["Комп."]>=20].head(30)["ФП"].tolist() if len(ct_seg) else []
    tidx = [i for i,n in enumerate(sc) if n in top_i]
    triple_res = []
    for i, j, k in combinations(tidx, 3):
        m = (X[:,i]==1) & (X[:,j]==1) & (X[:,k]==1)
        n = m.sum()
        if n < 10: continue
        nd = y[m].sum()
        if nd < 3: continue
        triple_res.append({"Комб.": f"{names[i]} + {names[j]} + {names[k]}",
                           "N": n, "Деф.": nd, "DR%": round(nd/n*100,2),
                           "Lift": round(nd/n/br,2)})

    dp = pd.DataFrame(pair_res).sort_values("Lift", ascending=False) if pair_res else pd.DataFrame()
    dt = pd.DataFrame(triple_res).sort_values("Lift", ascending=False) if triple_res else pd.DataFrame()

    print(f"\n{'='*60}\n{seg}: пар={len(dp)}, троек={len(dt)}")
    if len(dp): display(dp.head(5).style.hide(axis="index").bar(subset=["Lift"], color="salmon"))
    if len(dt): display(dt.head(5).style.hide(axis="index").bar(subset=["Lift"], color="salmon"))

## 11. Аномалии

In [ ]:
fp_100 = ct_all[(ct_all["DR%"]==100) & (ct_all["Комп."]>=2)]
fp_0 = ct_all[(ct_all["DR%"]==0) & (ct_all["Комп."]>=20)]
q99 = dataset["fp_unique"].quantile(0.99)
outliers = dataset[dataset["fp_unique"] > q99]
def_no_fp = companies_no_fp[companies_no_fp["default_flag"]==1]

print(f"100% DR (≥2 комп.): {len(fp_100)}")
if len(fp_100): display(fp_100.style.hide(axis="index"))
print(f"\n0% DR (≥20 комп.): {len(fp_0)}")
if len(fp_0): display(fp_0.style.hide(axis="index"))
print(f"\nРедкие (<5 комп.): {len(ct_all[ct_all['Комп.']<5])} из {len(ct_all)}")
print(f"Выбросы (>{q99:.0f} ФП): {len(outliers)}")
if len(outliers):
    display(outliers[["inn","segment","default_flag","fp_unique","fp_count"]]
            .sort_values("fp_unique", ascending=False).head(10).style.hide(axis="index"))
print(f"\nДеф. без предикторов: {len(def_no_fp)} ({len(def_no_fp)/n_def_companies:.1%})")

## 12. Lead time: первый ФП → дефолт

In [ ]:
fp_pd = fp_predictors[fp_predictors["default_flag"]==1].copy()
first_fp = fp_pd.groupby(["inn","segment"]).agg(first_fp_date=("dt","min")).reset_index()
first_fp = first_fp.merge(defaults[["inn","segment","default_date"]], on=["inn","segment"])
first_fp["lead_days"] = (first_fp["default_date"] - first_fp["first_fp_date"]).dt.days

print(f"{'Сегм.':<8} {'N':<8} {'Средн.':<10} {'Медиана':<10} {'Мин':<8} {'Макс'}")
print("=" * 48)
for seg in SEG_ORDER + ["ВСЕ"]:
    s = first_fp if seg == "ВСЕ" else first_fp[first_fp["segment"]==seg]
    if not len(s): continue
    d = s["lead_days"]
    print(f"{seg:<8} {len(s):<8,} {d.mean():<10.0f} {d.median():<10.0f} {d.min():<8} {d.max()}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, seg in enumerate(SEG_ORDER):
    s = first_fp[first_fp["segment"]==seg]["lead_days"]
    if not len(s): axes[i].set_title(f"{seg}: —"); continue
    axes[i].hist(s, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
    axes[i].axvline(s.median(), color="tomato", ls="--", lw=2,
                    label=f"Мед. {s.median():.0f} дн.")
    axes[i].set_title(seg, fontweight="bold")
    axes[i].set_xlabel("Дни"); axes[i].legend(fontsize=8)
plt.suptitle("Lead time: первый ФП → дефолт", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 13. Ускорение ФП перед дефолтом

In [ ]:
fp_pd2 = fp_predictors[fp_predictors["default_flag"]==1].copy()
fp_pd2 = fp_pd2.merge(defaults[["inn","segment","default_date"]], on=["inn","segment"])
fp_pd2["days_before"] = (fp_pd2["default_date"] - fp_pd2["dt"]).dt.days
fp_pd2["window"] = pd.cut(fp_pd2["days_before"],
    bins=[-1, 90, 180, fp_pd2["days_before"].max()+1],
    labels=["0-3 мес.", "3-6 мес.", "6+ мес."])

accel = (fp_pd2.groupby(["segment","inn","window"]).size()
         .reset_index(name="n").groupby(["segment","window"])["n"].mean().unstack())
display(accel.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
accel.reindex(columns=["6+ мес.","3-6 мес.","0-3 мес."]).plot(kind="bar", ax=ax, edgecolor="white")
ax.set_title("Ускорение ФП перед дефолтом", fontweight="bold")
ax.set_ylabel("Среднее ФП на компанию"); ax.set_xlabel("Сегмент")
ax.tick_params(axis="x", rotation=0); plt.tight_layout(); plt.show()

## 14. Порог ФП и default rate

In [ ]:
fig, axes = plt.subplots(1, len(SEG_ORDER)+1, figsize=(20, 5))
for idx, seg in enumerate(SEG_ORDER + ["ВСЕ"]):
    ax = axes[idx]
    s = dataset if seg == "ВСЕ" else dataset[dataset["segment"]==seg]
    mx = int(s["fp_unique"].quantile(0.95)) + 1
    dr_at = []
    for t in range(mx + 1):
        sub = s[s["fp_unique"] >= t]
        if not len(sub): break
        dr_at.append(sub["default_flag"].mean() * 100)
    ax.plot(range(len(dr_at)), dr_at, marker="o", markersize=4, color="tomato", lw=1.5)
    base = s["default_flag"].mean() * 100
    ax.axhline(base, color="gray", ls="--", lw=1, alpha=0.7)
    ax.set_title(seg, fontweight="bold")
    ax.set_xlabel("Мин. уник. ФП"); ax.set_ylabel("DR %"); ax.grid(True, alpha=0.3)
plt.suptitle("Default rate vs количество ФП", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

for seg in SEG_ORDER:
    s = dataset[dataset["segment"]==seg]
    print(f"{seg}:")
    for t in [0,1,3,5,10,15,20]:
        sub = s[s["fp_unique"]>=t]
        if not len(sub): break
        print(f"  ≥{t:>2}: {len(sub):>6,} комп., DR={sub['default_flag'].mean()*100:.2f}%")
    print()

## 15. Стабильность Lift по годам

In [ ]:
ds_y = dataset_bin.copy()
ds_y["year"] = ds_y["default_date"].dt.year

for seg in SEG_ORDER:
    ct = cross_tables.get(seg, pd.DataFrame())
    top = ct[ct["Деф."]>=3].head(20)["ФП"].tolist() if len(ct) else []
    if not top: continue
    sd = ds_y[ds_y["segment"]==seg]

    rows = []
    for fp in top:
        if fp not in sd.columns: continue
        row = {"ФП": fp}
        br = sd["default_flag"].mean()
        nw = (sd[fp]==1).sum(); nd = sd.loc[sd[fp]==1,"default_flag"].sum()
        row["Lift"] = round(nd/nw/br,2) if nw>0 and br>0 else 0
        for y in [2023,2024,2025]:
            ys = sd[(sd["year"]==y)|(sd["default_flag"]==0)]
            bry = ys["default_flag"].mean()
            ny = (ys[fp]==1).sum(); ndy = ys.loc[ys[fp]==1,"default_flag"].sum()
            row[f"{y}"] = round(ndy/ny/bry,2) if ny>0 and bry>0 else 0
        rows.append(row)

    print(f"\n{seg}: Lift по годам")
    display(pd.DataFrame(rows).style.hide(axis="index"))

## 16. Примеры дефолтных компаний

In [ ]:
for seg in SEG_ORDER:
    sd = defaults[defaults["segment"]==seg].sort_values("default_date")
    inns = sd["inn"].head(3).tolist()
    if not inns: continue
    print(f"\n{'='*70}\n{seg}")
    seg_df = df[df["segment"]==seg]
    for inn in inns:
        idf = seg_df[seg_df["inn"]==inn].sort_values("dt")
        dd = sd[sd["inn"]==inn]["default_date"].iloc[0]
        da = DEFAULT_ALL[seg]
        print(f"\n  {inn} | дефолт {dd:%d.%m.%Y} | {len(idf)} записей")
        for _, r in idf.head(20).iterrows():
            is_d = r["fp_num"] in da
            lbl = "ДЕФОЛТ" if is_d else ("ДО" if r["dt"] < dd else "ПОСЛЕ")
            print(f"    {r['IDENTIFICATION_DATE']}  {r['fp_num']:<12} ({r['TYPE_FP']})  [{lbl}]")

## 17. Сводка и сохранение

In [ ]:
for seg in SEG_ORDER:
    c = companies[companies["segment"]==seg]
    n, nd = len(c), c["default_flag"].sum()
    ct = cross_tables.get(seg, pd.DataFrame())
    hl = len(ct[(ct["Lift"]>2)&(ct["Деф."]>=3)]) if len(ct) else 0
    sd = dataset[dataset["segment"]==seg]
    ad = sd[sd["default_flag"]==1]["fp_unique"].mean() if nd else 0
    an = sd[sd["default_flag"]==0]["fp_unique"].mean()
    print(f"{seg}: {n:,} ИНН, {nd:,} деф. ({nd/n*100:.1f}%), "
          f"ср.ФП деф={ad:.1f}/недеф={an:.1f}, Lift>2: {hl}")

dataset.to_pickle(f"{DATA_DIR}/dataset_default_logic.pkl")
with pd.ExcelWriter(f"{DATA_DIR}/eda_default_logic.xlsx", engine="openpyxl") as w:
    ct_all.to_excel(w, sheet_name="Общая", index=False)
    for seg in SEG_ORDER:
        ct = cross_tables.get(seg)
        if ct is not None and len(ct):
            ct.to_excel(w, sheet_name=seg, index=False)
print(f"\nСохранено: dataset_default_logic.pkl, eda_default_logic.xlsx")